In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/boston-housing-dataset/BostonHousing.csv


In [3]:
import os

os.listdir('/kaggle/input/boston-housing-dataset/')


['BostonHousing.csv']

In [4]:
df = pd.read_csv('/kaggle/input/boston-housing-dataset/BostonHousing.csv')

In [30]:
print(df.isna().sum())

crim       0
zn         0
indus      0
chas       0
nox        0
rm         5
age        0
dis        0
rad        0
tax        0
ptratio    0
b          0
lstat      0
medv       0
dtype: int64


In [5]:
df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [11]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

In [31]:
class BostonDataset(Dataset):
    def __init__(self, df):

        df = df.fillna(df.mean())

        X = df.drop(columns=['medv']).values
        y = df['medv'].values

        mean = X.mean(axis=0)
        std = X.std(axis=0)

        std[std == 0] = 1
        
        self.X = torch.tensor((X - mean) / std, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1,1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [13]:
class LinearRegressionModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)

In [46]:
class RegressionMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)


In [15]:
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=100):
        for epoch in range(epochs):
            model.train()
            for X_batch, y_batch in train_loader:
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()


            if (epoch+1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Train Loss: {loss.item():.4f}")

In [54]:
dataset = BostonDataset(df)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=True)

#model = LinearRegressionModel(n_features=13)
model = RegressionMLP(n_features = 13)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

train_model(model, train_loader, test_loader, criterion, optimizer, epochs=500)

Epoch 10/500, Train Loss: 4.3065
Epoch 20/500, Train Loss: 25.9551
Epoch 30/500, Train Loss: 8.1196
Epoch 40/500, Train Loss: 4.5940
Epoch 50/500, Train Loss: 0.9811
Epoch 60/500, Train Loss: 4.6043
Epoch 70/500, Train Loss: 1.4919
Epoch 80/500, Train Loss: 1.5651
Epoch 90/500, Train Loss: 3.1060
Epoch 100/500, Train Loss: 3.3547
Epoch 110/500, Train Loss: 12.9992
Epoch 120/500, Train Loss: 23.6386
Epoch 130/500, Train Loss: 7.1178
Epoch 140/500, Train Loss: 1.6293
Epoch 150/500, Train Loss: 15.8559
Epoch 160/500, Train Loss: 8.5595
Epoch 170/500, Train Loss: 4.2500
Epoch 180/500, Train Loss: 4.3919
Epoch 190/500, Train Loss: 1.1823
Epoch 200/500, Train Loss: 5.2318
Epoch 210/500, Train Loss: 8.9058
Epoch 220/500, Train Loss: 5.4228
Epoch 230/500, Train Loss: 0.8325
Epoch 240/500, Train Loss: 11.8314
Epoch 250/500, Train Loss: 4.6943
Epoch 260/500, Train Loss: 17.0739
Epoch 270/500, Train Loss: 1.0911
Epoch 280/500, Train Loss: 3.1041
Epoch 290/500, Train Loss: 3.0250
Epoch 300/500, Tr

In [55]:
model.eval()
with torch.no_grad():
    total_loss = 0
    for X_batch, y_batch in test_loader:
        y_pred = model(X_batch)
        total_loss += criterion(y_pred, y_batch).item()
    avg_loss = total_loss / len(test_loader)

print(f"Test Loss: {avg_loss:.4f}")

Test Loss: 6.7442


In [56]:
sample_X, sample_y = next(iter(train_loader))
with torch.no_grad():
    print("Initial predictions:", model(sample_X)[:5].view(-1))
    print("Actual targets:", sample_y[:5].view(-1))


Initial predictions: tensor([31.9892, 18.4698, 18.7942, 31.6738])
Actual targets: tensor([33.1000, 19.8000, 19.3000, 32.2000])


In [28]:
print("Any NaNs in X?", torch.isnan(dataset.X).any().item())
print("Any NaNs in y?", torch.isnan(dataset.y).any().item())


Any NaNs in X? True
Any NaNs in y? False
